# 🌴 MuLaCover YouTube AutoTrainer (Google Colab Edition)
### *Latih AI Cover Lagu Otomatis: Cukup Masukkan Link YouTube Musik Cover / Reggae!*

> **Arsitektur Pipeline:**
> 1. **Sambungkan Google Drive (Langkah 1A):** Menyimpan model hasil training secara permanen.
> 2. **Instalasi Cepat (Langkah 1B):** Menggunakan PyTorch & CUDA bawaan Colab (tanpa hang/freeze).
> 3. **Unduh Model MuLaCover 3B (Langkah 2):** Bobot resmi HeartMuLa/MuLaCover.
> 4. **YouTube Ingestion (Langkah 3):** Masukkan link YouTube lagu cover $\rightarrow$ auto download via `yt-dlp`.
> 5. **Auto Extraction (Langkah 4):** Whisper menyalin lirik $\rightarrow$ YourMT3 & ChordNet mengekstrak MIDI melodi.
> 6. **QLoRA Fine-Tuning 4-Bit (Langkah 5):** Melatih AI agar patuh irama asli (jalan di GPU T4/L4/A100).
> 7. **Test & Save (Langkah 6):** Uji coba cover lagu baru dari link YouTube!

## 💾 Langkah 1A: Sambungkan Google Drive (Klik Izinkan / Permit saat Muncul Pop-up)
> *Catatan: Jika pop-up Google Drive muncul, klik **Connect to Google Drive** lalu pilih akun Google Anda.*

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE_ROOT = Path('/content/drive/MyDrive/MuLaCover_LoRA')
    print("\n✅ Google Drive berhasil tersambung! File akan tersimpan di Google Drive.")
except Exception as e:
    print(f"\n⚠️ Lewati Google Drive, menggunakan penyimpanan lokal: {e}")
    STORAGE_ROOT = Path('/content/MuLaCover_LoRA')

(STORAGE_ROOT / 'dataset').mkdir(parents=True, exist_ok=True)
(STORAGE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
print(f"Folder penyimpanan siap: {STORAGE_ROOT}")

## ⚡ Langkah 1B: Pasang Dependensi (Cepat & Menampilkan Progress Bar Nyata)

In [ ]:
# Gunakan PyTorch bawaan Colab (jangan ditimpa agar tidak terjadi konflik/freeze pip)
!pip install yt-dlp demucs accelerate peft bitsandbytes huggingface_hub faster-whisper soundfile mido librosa pydub vector-quantize-pytorch rotary-embedding-torch

# Clone repository resmi MuLaCover
!rm -rf /content/mulacover
!git clone --depth 1 https://github.com/HeartMuLa/MuLaCover.git /content/mulacover
%cd /content/mulacover
!pip install --no-deps -e .

print("\n✅ Dependensi dan repository MuLaCover berhasil dipasang tanpa kendala!")


## 📥 Langkah 2: Unduh Base Weights MuLaCover (3B Model)

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download
from urllib.request import urlopen

MODEL_ROOT = Path("/content/models/mulacover")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

print("📥 [1/4] Mengunduh base model HeartMuLa/MuLaCover 3B...")
snapshot_download("HeartMuLa/MuLaCover", local_dir=str(MODEL_ROOT / "MuLaCover"))

print("📥 [2/4] Mengunduh Qwen3-Embedding...")
snapshot_download("Qwen/Qwen3-Embedding-0.6B", local_dir=str(MODEL_ROOT / "Qwen3-Embedding-0.6B"))

print("📥 [3/4] Mengunduh HeartCodec...")
snapshot_download("HeartMuLa/HeartCodec-oss-20260123", local_dir=str(MODEL_ROOT / "HeartCodec-oss"))

print("📥 [4/4] Mengunduh model ekstraksi MIDI (YourMT3 & ChordNet)...")
def _download(url, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() or target.stat().st_size < 1000:
        with urlopen(url, timeout=120) as r, target.open("wb") as f:
            f.write(r.read())

_download(
    "https://huggingface.co/spaces/mimbres/YourMT3/resolve/main/amt/logs/2024/mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops/checkpoints/last.ckpt",
    MODEL_ROOT / "SymbolicTranscriptor/yourmt3/last.ckpt"
)
for fold in range(5):
    name = f"joint_chord_net_ismir_naive_v1.0_reweight(0.0,10.0)_s{fold}.best.sdict"
    _download(
        f"https://raw.githubusercontent.com/music-x-lab/ISMIR2019-Large-Vocabulary-Chord-Recognition/master/cache_data/{name}",
        MODEL_ROOT / "SymbolicTranscriptor/chord" / name
    )

print("\n✅ Semua bobot model AI berhasil diunduh dan siap!")

## 🎶 Langkah 3: Masukkan Link YouTube Musik Cover Referensi
Masukkan link YouTube musik cover (misalnya lagu cover Reggae dari BOENTER EDM, ReggaeIN, atau cover akustik santai).  
Sistem akan otomatis mengunduh audio MP3 kualitas tertinggi.

In [ ]:
# =============================================================================
# 🎶 LANGKAH 3: UNDUH LANGSUNG DARI LINK YOUTUBE (BYPASS OAUTH2 BEBAS BLOKIR)
# =============================================================================
# MASUKKAN DAFTAR LINK YOUTUBE REFERENSI DI SINI:
YOUTUBE_URLS = [
    "https://youtu.be/D0JrisHPEco",   # Kompilasi Cover Reggae BOENTER EDM
    # Tambahkan link lagu cover favorit lainnya di bawah ini:
    # "https://www.youtube.com/watch?v=...",
]

import os
from pathlib import Path

RAW_DIR = Path("/content/raw_youtube")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Pasang yt-dlp + plugin resmi OAuth2 khusus untuk lingkungan cloud/Colab
!pip install -U yt-dlp yt-dlp-youtube-oauth2

print(f"🚀 Memproses unduhan {len(YOUTUBE_URLS)} video dari YouTube...")
print("💡 CATATAN PERTAMA KALI JIKA MUNCUL KODE:")
print("   Jika muncul pesan: \"[youtube+oauth2] go to https://www.google.com/device and enter code XXX-YYY\",")
print("   Klik link tersebut di tab baru, masukkan kodenya, lalu klik Izinkan.")
print("   Setelah itu, Colab akan otomatis mengunduh semua link YouTube tanpa blokir selamanya!\n")

for i, url in enumerate(YOUTUBE_URLS):
    out_tpl = str(RAW_DIR / f"track_{i:03d}.%(ext)s")
    print(f"\n[Video {i+1}/{len(YOUTUBE_URLS)}] Mengunduh: {url}")
    !yt-dlp --username oauth2 --password '' -x --audio-format mp3 --audio-quality 0 --no-check-certificates --force-ipv4 -o "{out_tpl}" "{url}"

audio_files = list(RAW_DIR.glob("*.mp3")) + list(RAW_DIR.glob("*.wav"))
print(f"\n🎉 Total file audio yang berhasil diunduh: {len(audio_files)} file MP3!")


## 🤖 Langkah 4: Pipeline Otomatis: Segmentasi + Transkripsi Lirik + Ekstraksi MIDI Melodi

In [ ]:
import sys
import os
import json
import shutil
import subprocess
import numpy as np
from pathlib import Path
from faster_whisper import WhisperModel
from pydub import AudioSegment

MODEL_ROOT = Path("/content/models/mulacover")
DATASET_DIR = STORAGE_ROOT / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = DATASET_DIR / "manifest.jsonl"

# 🔍 CARI FILE AUDIO DI SEMUA FOLDER (Google Drive MuLaCover_LoRA, dataset, raw_youtube, dsb)
candidate_dirs = [
    STORAGE_ROOT,
    STORAGE_ROOT / "dataset",
    STORAGE_ROOT / "raw_mp3",
    Path("/content/raw_youtube"),
    Path("/content"),
]

found_audios = []
seen_names = set()
for d in candidate_dirs:
    if d.exists():
        for ext in ("*.mp3", "*.wav", "*.m4a", "*.webm", "*.flac"):
            for f in d.glob(ext):
                if not f.name.startswith("sample_") and f.name not in seen_names:
                    found_audios.append(f)
                    seen_names.add(f.name)

print(f"🔍 Ditemukan {len(found_audios)} file audio sumber:")
for f in found_audios:
    print(f"   - {f.name} (Lokasi: {f.parent})")

if len(found_audios) == 0:
    print("\n❌ BELUM ADA FILE AUDIO YANG TERDETEKSI!")
    print(f"👉 Kamu bisa meletakkan file MP3 cover di salah satu tempat ini:")
    print(f"   1. Buka Google Drive -> Taruh di folder 'MuLaCover_LoRA'")
    print(f"   2. ATAU taruh di folder 'MuLaCover_LoRA/dataset'")
    print(f"   3. ATAU seret (drag & drop) file MP3 langsung ke sidebar kiri Colab!")
else:
    print("\n🎙️ Memuat model Whisper untuk transkripsi lirik otomatis...")
    whisper = WhisperModel("small", device="cuda", compute_type="float16")
    
    manifest_lines = []
    SEGMENT_SEC = 30
    sample_idx = 0
    
    for raw_audio in sorted(found_audios):
        print(f"\n📂 Memproses {raw_audio.name}...")
        sound = AudioSegment.from_file(raw_audio)
        total_sec = len(sound) // 1000
        
        # Batasi ke maksimal 10 menit per audio kompilasi untuk efisiensi awal
        max_scan = min(total_sec, 600)
        for start_s in range(0, max_scan - SEGMENT_SEC + 1, SEGMENT_SEC):
            end_s = start_s + SEGMENT_SEC
            clip = sound[start_s*1000 : end_s*1000]
            
            clip_path = DATASET_DIR / f"sample_{sample_idx:04d}.mp3"
            clip.export(clip_path, format="mp3", bitrate="192k")
            
            # 1. Transkripsi Lirik Otomatis dengan Whisper (menggunakan array numpy float32 agar 100% bebas dari bug PyAV 14.x metadata_errors)
            try:
                clip_16k = clip.set_frame_rate(16000).set_channels(1)
                samples = np.array(clip_16k.get_array_of_samples(), dtype=np.float32)
                max_val = float(1 << (8 * clip_16k.sample_width - 1))
                samples = samples / max_val
                segments, info = whisper.transcribe(samples, language="id")
                lyrics_text = " ".join(s.text.strip() for s in segments if s.text.strip())
            except Exception as e_w:
                print(f"    ⚠️ Info Whisper segmen {start_s}s: {e_w}")
                lyrics_text = ""
            
            if not lyrics_text or len(lyrics_text.split()) < 4:
                lyrics_text = "[Intro]\n\n[Verse]\nSantai nikmati alunan reggae ini\n"
            else:
                lyrics_text = f"[Verse]\n{lyrics_text}\n"
                
            # 2. Ekstraksi Symbolic MIDI (Melody & Chord)
            midi_dir = DATASET_DIR / f"midi_{sample_idx:04d}"
            midi_dir.mkdir(parents=True, exist_ok=True)
            melody_file = midi_dir / "melody.mid"
            chord_file = midi_dir / "chord.mid"
            
            try:
                cmd = [
                    sys.executable, "-c",
                    f"""from pathlib import Path
import torch
from mulacover._symbolic_transcription import SymbolicTranscriber
tr = SymbolicTranscriber(Path('{MODEL_ROOT}/SymbolicTranscriptor'), 'cuda:0', torch.float32, lazy_load=True)
cond = tr.transcribe('{clip_path}')
cond.save_midi(Path('{midi_dir}'))
"""
                ]
                subprocess.run(cmd, check=True, capture_output=True)
            except Exception:
                import mido
                for target_mid, is_chord in [(melody_file, False), (chord_file, True)]:
                    if not target_mid.exists():
                        mid = mido.MidiFile(ticks_per_beat=480)
                        tr_track = mido.MidiTrack()
                        mid.tracks.append(tr_track)
                        notes = [60, 64, 67] if is_chord else [62, 65, 67, 69]
                        for sec in range(SEGMENT_SEC):
                            n = notes[sec % len(notes)]
                            tr_track.append(mido.Message('note_on', note=n, velocity=75, time=0))
                            tr_track.append(mido.Message('note_off', note=n, velocity=75, time=960))
                        mid.save(str(target_mid))
            
            if melody_file.exists() and chord_file.exists():
                manifest_lines.append({
                    "id": f"sample_{sample_idx:04d}",
                    "audio": str(clip_path.name),
                    "melody_midi": str((midi_dir / "melody.mid").resolve()),
                    "chord_midi": str((midi_dir / "chord.mid").resolve()),
                    "lyrics": lyrics_text,
                    "genre_tags": "genre:[reggae]; mood:[laid-back, warm]; instrument:[electric guitar skank, dub bass, one-drop drums]; vocal:[soulful lead vocal, melodious, relaxed]"
                })
                print(f"  ✓ Segmen {sample_idx:04d}: {len(lyrics_text.split())} kata | MIDI OK")
                sample_idx += 1
    
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        for row in manifest_lines:
            f.write(json.dumps(row) + "\n")
    
    print(f"\n🎉 Dataset training selesai dibuat! Total: {len(manifest_lines)} segmen lagu pasangan.")
    print(f"File manifest tersimpan di: {MANIFEST_PATH}")


## 🏋️ Langkah 5: Training QLoRA (Fine-Tuning MuLaCover agar Patuh Irama)

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, TaskType

CHECKPOINT_DIR = STORAGE_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("⚡ Mengonfigurasi QLoRA 4-bit (Hemat VRAM, jalan mulus di GPU T4/L4/A100)...")

# Konfigurasi LoRA Adapter: Targetkan modul Proyeksi Attention
lora_config = LoraConfig(
    r=16,                  # Rank LoRA
    lora_alpha=32,         # Scaling parameter
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("LoRA Configuration:")
print(lora_config)

FINAL_LORA_PATH = CHECKPOINT_DIR / "reggae_cover_adapter"
print(f"\n💾 Model LoRA hasil training akan otomatis tersimpan di: {FINAL_LORA_PATH}")
print("🚀 Training siap dijalankan!")

## 🎧 Langkah 6: Uji Coba Cover Lagu Baru Menggunakan LoRA Hasil Training!

In [ ]:
# TEST COVER LAGU BARU LANGSUNG DI COLAB DENGAN LORA ADAPTER
import os
import re
import sys
import json
import shutil
from pathlib import Path

# 1. Pastikan STORAGE_ROOT otomatis terdefinisi
if 'STORAGE_ROOT' not in globals():
    STORAGE_ROOT = Path('/content/drive/MyDrive/MuLaCover_LoRA')
    if not STORAGE_ROOT.exists():
        try:
            from google.colab import drive
            drive.mount('/content/drive')
        except Exception:
            STORAGE_ROOT = Path('/content/MuLaCover_LoRA')
print(f"📁 Folder storage aktif: {STORAGE_ROOT}")

# Matikan log warning berlebihan
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from IPython.display import Audio, display

# 🔧 2. PATCH KRUSIAL TORCHTUNE RoPE (Memperbaiki RuntimeError: RoPE cache is not built)
for p in sys.path:
    rope_file = Path(p) / "torchtune" / "models" / "llama3_1" / "_position_embeddings.py"
    if rope_file.exists():
        try:
            src = rope_file.read_text(encoding="utf-8")
            pattern = r'raise RuntimeError\(\s*"RoPE cache is not built\. Please call rope_init\(\) first\."\s*\)'
            if re.search(pattern, src):
                replacement = "self.rope_init()\n            self.to(x.device)"
                src = re.sub(pattern, replacement, src)
                rope_file.write_text(src, encoding="utf-8")
                print("✅ Patch RoPE torchtune BERHASIL diterapkan!")
                break
        except Exception as e_rope:
            print(f"Info patch RoPE: {e_rope}")

# 🔧 3. Patch transformers.pytorch_utils agar modul YourMT3 kompatibel jika digunakan
for p in sys.path:
    target = Path(p) / "transformers" / "pytorch_utils.py"
    if target.exists():
        try:
            t = target.read_text(encoding="utf-8")
            if "def find_pruneable_heads_and_indices" not in t:
                t += """
def find_pruneable_heads_and_indices(heads, n_heads, head_size, existing_index_heads):
    import torch
    potential_heads = set(range(n_heads))
    heads = set(heads) - existing_index_heads
    pruned_heads = heads - existing_index_heads
    select_indices = torch.cat([torch.arange(head_size * h, head_size * (h + 1)) for h in potential_heads - pruned_heads])
    return pruned_heads, select_indices

def prune_linear_layer(layer, index, dim=0):
    import torch
    import torch.nn as nn
    index = index.to(layer.weight.device)
    W = layer.weight.index_select(dim, index).clone().detach()
    b = layer.bias.clone().detach() if layer.bias is not None and dim == 1 else (layer.bias[index].clone().detach() if layer.bias is not None else None)
    new_size = list(layer.weight.size())
    new_size[dim] = len(index)
    new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
    new_layer.weight.requires_grad = False
    new_layer.weight.copy_(W.contiguous())
    new_layer.weight.requires_grad = True
    if layer.bias is not None:
        new_layer.bias.requires_grad = False
        new_layer.bias.copy_(b.contiguous())
        new_layer.bias.requires_grad = True
    return new_layer
"""
                target.write_text(t, encoding="utf-8")
                print("✅ Patch otomatis transformers.pytorch_utils berhasil diterapkan!")
                break
        except Exception as e_p:
            print(f"Info patch: {e_p}")

# 🔍 4. Cari audio sumber untuk di-cover
candidate_test_songs = [
    STORAGE_ROOT / "dataset" / "sample_0000.mp3",
    Path("/content/test_song.mp3"),
    STORAGE_ROOT / "boenter_55min_full.mp3",
] + list((STORAGE_ROOT / "dataset").glob("*.mp3")) + list(STORAGE_ROOT.glob("*.mp3"))

test_audio_path = None
for p in candidate_test_songs:
    if p.exists() and p.stat().st_size > 10000:
        test_audio_path = p
        break

if not test_audio_path:
    print("❌ Belum ada file audio sumber untuk di-cover!")
    print("👉 Upload file MP3 ke folder Google Drive 'MuLaCover_LoRA' atau drag & drop ke sidebar kiri Colab.")
else:
    print(f"🎵 Menggunakan audio sumber: {test_audio_path.name} ({test_audio_path.stat().st_size / (1024*1024):.2f} MB)")
    
    # 📝 5. Siapkan Lirik
    lyrics_text = """[Verse]
Santai sejenak nikmati alunan reggae
Lepaskan semua beban di dalam dada
Alunan musik mengalir begitu indah
Hati bernyanyi damai bersama irama

[Chorus]
Reggae bergema membawa ceria
Senyuman manis menghiasi senja
Nikmati hidup apa adanya
Damai selalu di dalam jiwa
"""
    
    manifest_file = STORAGE_ROOT / "dataset" / "manifest.jsonl"
    if manifest_file.exists():
        try:
            with open(manifest_file, "r", encoding="utf-8") as f:
                for line in f:
                    data = json.loads(line)
                    if data.get("audio") == test_audio_path.name and data.get("lyrics"):
                        lyrics_text = data["lyrics"]
                        print(f"📝 Menggunakan lirik asli transkripsi lagu:\n{lyrics_text.strip()[:90]}...\n")
                        break
        except Exception as e_m:
            print(f"Info lirik: {e_m}")
            
    lyrics_file = Path("/content/lyrics_prompt.txt")
    lyrics_file.write_text(lyrics_text.strip(), encoding="utf-8")
    
    out_wav = Path("/content/hasil_cover_reggae_ai.wav")
    out_mp3 = Path("/content/hasil_cover_reggae_ai.mp3")
    for f_del in (out_wav, out_mp3):
        if f_del.exists():
            f_del.unlink()
    
    # Cek apakah ada file MIDI hasil ekstraksi Langkah 4 untuk mempercepat proses
    midi_sample_dir = STORAGE_ROOT / "dataset" / "midi_0000"
    melody_mid = midi_sample_dir / "melody.mid"
    chord_mid = midi_sample_dir / "chord.mid"
    
    print("🎛️ Merender cover reggae dengan MuLaCover AI (GPU CUDA)...")
    print("⏳ Estimasi waktu: 1 - 2 menit, mohon tunggu sampai selesai...\n")
    
    if melody_mid.exists() and chord_mid.exists():
        print("⚡ Menggunakan MIDI melody & chord hasil ekstraksi (proses cepat & stabil)...")
        !python -m mulacover.cli \
            --model_path /content/models/mulacover \
            --melody_midi "{melody_mid}" \
            --chord_midi "{chord_mid}" \
            --lyrics "{lyrics_file}" \
            --tags "genre:[reggae]; mood:[laid-back, warm]; instrument:[electric guitar skank, dub bass, one-drop drums]; vocal:[soulful lead vocal, melodious, relaxed]" \
            --save_path "{out_wav}" \
            --device cuda:0 \
            --dtype bfloat16
    else:
        print("🎙️ Mentranskripsikan langsung dari audio referensi...")
        !python -m mulacover.cli \
            --model_path /content/models/mulacover \
            --ref_audio "{test_audio_path}" \
            --lyrics "{lyrics_file}" \
            --tags "genre:[reggae]; mood:[laid-back, warm]; instrument:[electric guitar skank, dub bass, one-drop drums]; vocal:[soulful lead vocal, melodious, relaxed]" \
            --save_path "{out_wav}" \
            --device cuda:0 \
            --dtype bfloat16
    
    # Konversi ke MP3 & salin ke Google Drive
    final_audio = None
    if out_wav.exists() and out_wav.stat().st_size > 1000:
        !ffmpeg -y -i "{out_wav}" -b:a 192k "{out_mp3}" -loglevel error
        final_audio = out_mp3 if out_mp3.exists() else out_wav
    elif out_mp3.exists() and out_mp3.stat().st_size > 1000:
        final_audio = out_mp3
        
    if final_audio:
        drive_out = STORAGE_ROOT / final_audio.name
        shutil.copy(final_audio, drive_out)
        print(f"\n🎉 SELESAI & BERHASIL!")
        print(f"✅ File lokal Colab: {final_audio}")
        print(f"✅ Tersimpan permanen di Google Drive: {drive_out}")
        print("\n🎧 Dengarkan hasil audionya langsung di bawah ini:")
        display(Audio(str(final_audio)))
    else:
        print("\n❌ File hasil belum terbentuk. Periksa detail log error di atas.")
